# API Investigation for Market Analysis System

**Objetivo:** Investigar APIs disponíveis para coleta de dados de fundos de investimento, benchmarks e notícias.

**Requisitos do sistema:**
- Dados de performance de fundos do Nubank
- Benchmarks: SELIC, CDI, Tesouro
- Notícias relacionadas ao Nubank
- Execução diária às 9h
- Geração de PDF com análise comparativa

## APIs a investigar:

1. **APIs Oficiais**
   - BCB (Banco Central) - SELIC, CDI
   - Tesouro Direto - Tesouro IPCA+
   - CVM - Dados de fundos

2. **APIs de Notícias**
   - Google News API
   - News API
   - RSS feeds

3. **Fallbacks**
   - MaisRetorno (scraping se necessário)
   - InfoMoney
   - Yahoo Finance

## Setup

In [ ]:
import requests
import pandas as pd
import json
from datetime import datetime, timedelta
import time
from typing import Dict, List, Optional

# Para debugging
import pprint
pp = pprint.PrettyPrinter(indent=2)

## 1. Banco Central do Brasil (BCB) - SELIC e CDI

API oficial do BCB para dados econômicos.

In [ ]:
# BCB API - Sistema de Séries Temporais (SGS)
# Documentação: https://dadosabertos.bcb.gov.br/dataset/20542-selic-taxa-de-juros

def test_bcb_selic():
    """
    Testa API do BCB para taxa SELIC
    Série: 432 - Taxa SELIC
    """
    
    # Últimos 30 dias
    end_date = datetime.now().strftime('%d/%m/%Y')
    start_date = (datetime.now() - timedelta(days=30)).strftime('%d/%m/%Y')
    
    url = f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.432/dados?formato=json&dataInicial={start_date}&dataFinal={end_date}"
    
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        
        data = response.json()
        
        print("✅ BCB SELIC API - Success")
        print(f"Status Code: {response.status_code}")
        print(f"Records found: {len(data)}")
        
        if data:
            print("\nSample data:")
            pp.pprint(data[-3:])  # últimos 3 registros
            
        return data
        
    except Exception as e:
        print(f"❌ BCB SELIC API - Error: {e}")
        return None

# Test SELIC
selic_data = test_bcb_selic()

In [ ]:
def test_bcb_cdi():
    """
    Testa API do BCB para taxa CDI
    Série: 4389 - Taxa CDI
    """
    
    # Últimos 30 dias
    end_date = datetime.now().strftime('%d/%m/%Y')
    start_date = (datetime.now() - timedelta(days=30)).strftime('%d/%m/%Y')
    
    url = f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.4389/dados?formato=json&dataInicial={start_date}&dataFinal={end_date}"
    
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        
        data = response.json()
        
        print("✅ BCB CDI API - Success")
        print(f"Status Code: {response.status_code}")
        print(f"Records found: {len(data)}")
        
        if data:
            print("\nSample data:")
            pp.pprint(data[-3:])  # últimos 3 registros
            
        return data
        
    except Exception as e:
        print(f"❌ BCB CDI API - Error: {e}")
        return None

# Test CDI
cdi_data = test_bcb_cdi()

## 2. Tesouro Direto - Tesouro IPCA+

API oficial do Tesouro Nacional.

In [ ]:
def test_tesouro_direto():
    """
    Testa API do Tesouro Direto
    """
    
    url = "https://www.tesourotransparente.gov.br/ckan/api/3/action/datastore_search"
    
    # Parâmetros para buscar dados do Tesouro IPCA+
    params = {
        'resource_id': 'af63adf4-68a0-4f36-9aa5-0c8c89d4e8ff',  # ID do dataset de preços
        'limit': 10
    }
    
    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        
        data = response.json()
        
        print("✅ Tesouro Direto API - Success")
        print(f"Status Code: {response.status_code}")
        
        if data.get('success'):
            records = data['result']['records']
            print(f"Records found: {len(records)}")
            
            if records:
                print("\nSample data:")
                pp.pprint(records[:3])  # primeiros 3 registros
                
        return data
        
    except Exception as e:
        print(f"❌ Tesouro Direto API - Error: {e}")
        return None

# Test Tesouro Direto
tesouro_data = test_tesouro_direto()

## 3. CVM - Dados de Fundos de Investimento

API da CVM para dados oficiais de fundos.

In [ ]:
def test_cvm_fundos():
    """
    Testa dados da CVM para fundos de investimento
    Dados públicos da CVM
    """
    
    # CVM disponibiliza dados via FTP e arquivos CSV
    # Vamos tentar acessar informações cadastrais dos fundos
    
    url = "http://dados.cvm.gov.br/dados/FI/CAD/DADOS/cad_fi.csv"
    
    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        
        print("✅ CVM Fundos Data - Success")
        print(f"Status Code: {response.status_code}")
        print(f"Content size: {len(response.content)} bytes")
        
        # Vamos ler o CSV e procurar por fundos do Nubank
        from io import StringIO
        
        df = pd.read_csv(StringIO(response.text), sep=';', encoding='latin-1')
        
        print(f"Total de fundos: {len(df)}")
        print(f"Colunas: {list(df.columns)}")
        
        # Procurar fundos do Nubank
        nubank_funds = df[df['DENOM_SOCIAL'].str.contains('NUBANK|Nu', case=False, na=False)]
        
        print(f"\nFundos do Nubank encontrados: {len(nubank_funds)}")
        
        if len(nubank_funds) > 0:
            print("\nSample Nubank funds:")
            print(nubank_funds[['CNPJ_FUNDO', 'DENOM_SOCIAL']].head())
            
        return df
        
    except Exception as e:
        print(f"❌ CVM Fundos Data - Error: {e}")
        return None

# Test CVM
cvm_data = test_cvm_fundos()

## 4. APIs de Notícias

In [ ]:
def test_google_news_rss():
    """
    Testa RSS feed do Google News para notícias do Nubank
    """
    
    import xml.etree.ElementTree as ET
    
    # Google News RSS para busca "Nubank"
    url = "https://news.google.com/rss/search?q=Nubank&hl=pt-BR&gl=BR&ceid=BR:pt-419"
    
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        
        print("✅ Google News RSS - Success")
        print(f"Status Code: {response.status_code}")
        
        # Parse XML
        root = ET.fromstring(response.content)
        
        items = root.findall('.//item')
        print(f"Articles found: {len(items)}")
        
        if items:
            print("\nSample articles:")
            for i, item in enumerate(items[:3]):
                title = item.find('title').text if item.find('title') is not None else 'N/A'
                pub_date = item.find('pubDate').text if item.find('pubDate') is not None else 'N/A'
                print(f"{i+1}. {title} ({pub_date})")
                
        return items
        
    except Exception as e:
        print(f"❌ Google News RSS - Error: {e}")
        return None

# Test Google News RSS
news_data = test_google_news_rss()

In [ ]:
def test_newsapi():
    """
    Testa News API (requer API key)
    """
    
    # News API requer chave, vamos apenas testar se o endpoint está disponível
    url = "https://newsapi.org/v2/everything"
    
    params = {
        'q': 'Nubank',
        'language': 'pt',
        'sortBy': 'publishedAt',
        'pageSize': 5
    }
    
    try:
        # Sem API key para testar estrutura da resposta
        response = requests.get(url, params=params, timeout=10)
        
        print("📊 News API - Endpoint test")
        print(f"Status Code: {response.status_code}")
        
        if response.status_code == 401:
            print("⚠️  API Key required (expected)")
            print("Endpoint is accessible, would work with valid API key")
        else:
            data = response.json()
            pp.pprint(data)
            
        return response.status_code
        
    except Exception as e:
        print(f"❌ News API - Error: {e}")
        return None

# Test News API
newsapi_status = test_newsapi()

## 5. Alternative Data Sources

In [ ]:
def test_yahoo_finance():
    """
    Testa Yahoo Finance para dados financeiros
    """
    
    # Yahoo Finance não tem API oficial, mas vamos testar se conseguimos dados básicos
    # Usando CDI como exemplo (pode não funcionar)
    
    try:
        import yfinance as yf
        print("✅ yfinance library available")
        
        # Testar com algum ticker brasileiro
        ticker = "^BVSP"  # Ibovespa
        stock = yf.Ticker(ticker)
        hist = stock.history(period="5d")
        
        print(f"Yahoo Finance data for {ticker}:")
        print(f"Records: {len(hist)}")
        print(hist.head())
        
        return hist
        
    except ImportError:
        print("⚠️  yfinance library not installed")
        print("Would need: pip install yfinance")
        return None
    except Exception as e:
        print(f"❌ Yahoo Finance - Error: {e}")
        return None

# Test Yahoo Finance
yahoo_data = test_yahoo_finance()

## 6. MaisRetorno (Website Analysis)

In [ ]:
def test_maisretorno():
    """
    Testa acesso ao MaisRetorno para análise de estrutura
    """
    
    url = "https://maisretorno.com/fundos"
    
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }
    
    try:
        response = requests.get(url, headers=headers, timeout=10)
        
        print("📊 MaisRetorno - Accessibility test")
        print(f"Status Code: {response.status_code}")
        print(f"Content size: {len(response.content)} bytes")
        
        # Verificar se há conteúdo relevante
        content = response.text.lower()
        
        keywords = ['nubank', 'fundo', 'rentabilidade', 'cdi']
        found_keywords = [kw for kw in keywords if kw in content]
        
        print(f"Keywords found: {found_keywords}")
        
        if response.status_code == 200:
            print("✅ Website accessible, would require scraping")
        
        return response.status_code
        
    except Exception as e:
        print(f"❌ MaisRetorno - Error: {e}")
        return None

# Test MaisRetorno
maisretorno_status = test_maisretorno()

## Resumo dos Testes

Vamos compilar os resultados dos testes para avaliar viabilidade.

In [ ]:
# Compilation of results
def compile_results():
    """
    Compila resultados dos testes de APIs
    """
    
    results = {
        "BCB_SELIC": selic_data is not None,
        "BCB_CDI": cdi_data is not None,
        "Tesouro_Direto": tesouro_data is not None,
        "CVM_Fundos": cvm_data is not None,
        "Google_News_RSS": news_data is not None,
        "NewsAPI": newsapi_status == 401,  # 401 means API exists but needs key
        "Yahoo_Finance": yahoo_data is not None,
        "MaisRetorno": maisretorno_status == 200
    }
    
    print("\n" + "="*50)
    print("RESUMO DOS TESTES DE APIs")
    print("="*50)
    
    for api, success in results.items():
        status = "✅ FUNCIONANDO" if success else "❌ FALHOU"
        print(f"{api:20} {status}")
    
    print("\n" + "="*50)
    print("RECOMENDAÇÕES")
    print("="*50)
    
    recommendations = []
    
    if results["BCB_SELIC"] and results["BCB_CDI"]:
        recommendations.append("✅ Use BCB API para SELIC e CDI (dados oficiais)")
    
    if results["Tesouro_Direto"]:
        recommendations.append("✅ Use Tesouro Direto API para Tesouro IPCA+")
    
    if results["CVM_Fundos"]:
        recommendations.append("✅ Use CVM dados para fundos do Nubank")
    
    if results["Google_News_RSS"]:
        recommendations.append("✅ Use Google News RSS para notícias (gratuito)")
    
    if not any([results["BCB_SELIC"], results["CVM_Fundos"]]):
        recommendations.append("⚠️  Considere scraping como fallback")
    
    for rec in recommendations:
        print(rec)
    
    return results

# Compile final results
final_results = compile_results()

## Próximos Passos

Com base nos resultados dos testes:

1. **Implementar pipeline com APIs funcionais**
2. **Definir estrutura de dados unificada**
3. **Criar sistema de fallback para dados não disponíveis via API**
4. **Testar frequência de atualização das APIs**
5. **Implementar cache local para dados históricos**